# Dia 3 — A caixa trancada 🔐

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-03-instalar-programas](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-03-instalar-programas)

## 🎬 Missão

> A polícia apreendeu o notebook de um "atendente" da PixVelox e mandou uma
> cópia para o Núcleo: dois arquivos, `inventario.zip` e `evidencia.zip`.
> O segundo pede senha.
>
> — O dono jura que não lembra a senha — diz a Mestra Iara, colocando o
> café na mesa. — Mas quem anota rotina em arquivo de texto também anota
> senha em algum lugar. Só não abra nada antes de saber o que tem dentro.
>
> Bento, sem tirar os olhos da tela:
> — E instala o `unzip` primeiro. Seu terminal ainda nem tem.

**Sua tarefa:** descobrir a senha da `desafio/evidencia.zip` e ler a flag
`VAZANTE{...}` que está no relatório lá dentro.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### O que já está instalado? (dpkg -l)

```text
No Ubuntu/Debian, cada programa vem num "pacote" .deb. O `dpkg` é quem
instala e registra esses pacotes. `dpkg -l` lista TODOS os instalados.
São centenas, então mostramos só as primeiras linhas com `head`.
As letras "ii" no começo significam: desejado=instalar, estado=instalado.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "Pacotes instalados (início da lista)"
dpkg -l | head -n 12
echo "..."
echo "Total de pacotes instalados: $(dpkg -l | grep -c '^ii')"

### Perguntando por um pacote específico

```text
`dpkg -l nome` mostra só aquele pacote. `command -v` diz ONDE está o
programa (ou nada, se não existir). `dpkg -S` diz de qual pacote ele veio.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "O zip está aí?"
dpkg -l zip unzip 2>/dev/null | grep '^ii' || alerta "zip/unzip ainda não instalados"
command -v ls
dpkg -S "$(command -v ls)"     # o ls vem do pacote coreutils

### Instalando com apt

```text
O `apt` baixa pacotes da internet (dos repositórios do Ubuntu), resolve as
dependências e chama o dpkg por você. Instalar exige ser administrador,
por isso o `sudo` na frente:
    sudo apt update               # atualiza a lista de pacotes disponíveis
    sudo apt install zip unzip    # instala os dois
Aqui usamos a função `precisa` (de lib/comum.sh): ela só instala se o
programa ainda não existir — rodar duas vezes não faz mal.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "Instalando zip e unzip (se faltar)"
mostrar "sudo apt install zip unzip"
precisa zip
precisa unzip
zip -v | head -n 2
dica "Outros comandos úteis: apt search <palavra>, apt show <pacote>, sudo apt remove <pacote>"

### Detalhes de um pacote antes de instalar

```text
`apt-cache policy` mostra versão instalada e versão disponível.
`apt show` mostra a descrição. Bom para conferir antes de instalar algo.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "Informações do pacote zip"
apt-cache policy zip 2>/dev/null | head -n 3
apt show zip 2>/dev/null | grep -E '^(Package|Version|Description):' || true

### Programas Python: pip

```text
Muitas ferramentas de OSINT são escritas em Python e se instalam com pip.
  python3 -m pip install nome-do-pacote
Em Ubuntu recente, prefira `pipx install ferramenta` (isola cada uma) ou um
ambiente virtual (python3 -m venv). Aqui só conferimos a versão.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "pip"
python3 -m pip --version 2>/dev/null || alerta "pip não instalado (sudo apt install python3-pip)"

### Compactando: zip

```text
`zip destino arquivos...` cria (ou atualiza) um .zip.
A extensão .zip é posta automaticamente se você esquecer.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "Criando archive.zip"
mkdir -p treino
cp desafio/test1.txt desafio/test2.txt treino/
(
  cd treino || exit 1
  zip archive test1.txt test2.txt
  ls -l
)

### Olhando dentro do zip SEM extrair

```text
Essencial em investigação: você vê o que tem antes de abrir.
  zip -sf arquivo.zip     (show files) lista os nomes
  unzip -l arquivo.zip    lista com tamanho e data
  unzip -v arquivo.zip    ainda mais detalhes (método, CRC)
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "Listando sem extrair"
zip -sf treino/archive.zip
unzip -l treino/archive.zip

### Extraindo: unzip

```text
`unzip arquivo.zip -d pasta` extrai dentro da pasta indicada (cria se
não existir). Sem -d, extrai na pasta atual — e pode bagunçar tudo.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "Extraindo para uma pasta"
unzip -o treino/archive.zip -d treino/archive
ls -l treino/archive

### Zipando pastas inteiras e lendo um arquivo direto

```text
  zip -r      inclui subpastas (recursivo)
  unzip -p    imprime o conteúdo de um arquivo de dentro do zip, sem extrair
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "zip -r e unzip -p"
zip -rq treino/pasta.zip treino/archive
unzip -l treino/pasta.zip
unzip -p treino/archive.zip test1.txt

### Zip com senha

```text
  zip -e        pede a senha (interativo)
  zip -P senha  passa a senha na linha de comando (fica no histórico! só em treino)
  unzip -P senha arquivo.zip   extrai com a senha
Atenção: os NOMES dos arquivos de um zip com senha continuam visíveis.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "Zip com senha"
interativo zip -e treino/secreto.zip treino/test1.txt
zip -q -P treino123 treino/secreto.zip treino/test1.txt
unzip -l treino/secreto.zip
unzip -o -q -P treino123 treino/secreto.zip -d treino/aberto && ok "abriu com a senha"
unzip -o -q -P errada treino/secreto.zip -d treino/aberto2 2>/dev/null || alerta "senha errada: unzip recusa"

### Testando a integridade

```text
`unzip -t` testa se o zip está íntegro, sem extrair nada.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "Teste de integridade"
unzip -t treino/archive.zip

### Limpando

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "Limpando a pasta de treino"
rm -r treino
ls

### O inventário do notebook apreendido

```text
A evidência de hoje: dois zips. Um aberto, outro trancado.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-03-instalar-programas" && source "$L4O_HOME/lib/comum.sh"
titulo "Os zips do desafio"
ls -l desafio/*.zip
unzip -l desafio/evidencia.zip
dica "evidencia.zip pede senha. Olhe o inventario.zip com atenção — sem extrair."

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-03-instalar-programas" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-03-instalar-programas" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)